# Generate response from Language model 

In [1]:
import ollama # Used to load model .
from textwrap import dedent # Used for spacing problems in prompt .
from tabulate import tabulate # Used for creating a table for displaying models .
import subprocess # Used to start ollama server .
import time # For waiting .
import requests # Used to access ollama server .

In [2]:
# Used to initialize a language model and generate responses .
class LocalLLM:
    def __init__(self,model_name:str="gemma3:4b"): # Here is where the model loads .
        self.model_name=model_name
        self.process=self.ollama_server(process="start")

        if not self.is_model_available(self.model_name): # Checking model is available or valid .
            available = [m['model_name'] for m in self.available_models()]
            raise ValueError(
                f"Model '{model_name}' not available. "
                f"Available models: {available}"
            )

# The following function is used to start or stop ollama server .
    def ollama_server(self, process: str):
        if process == "start":
            try:
                requests.get("http://localhost:11434/api/tags", timeout=1) # Checking if ollama server is already started .
                print("Ollama already running")
                return "external"
            except:
                pass

            ollama_process = subprocess.Popen( # Starting ollama server if its not started .
                ["ollama", "serve"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
                shell=False
            )

            for _ in range(10): # Checking if sever started .
                try:
                    requests.get("http://localhost:11434/api/tags", timeout=1)
                    print("Ollama server started")
                    return ollama_process
                except:
                    time.sleep(1)

            raise RuntimeError("Ollama failed to start") # If server did not start after many tries than raising error .

        elif process == "stop": # Stopping ollama server .
            if isinstance(self.process, subprocess.Popen): # Checking if server was started using here .
                self.process.terminate()
                self.process.wait()
                print(print("Ollama server successfully stopped ."))
            else:
                print("Ollama was not started by this process") # If python server started externally then notifying it .

            return None

        else:
            raise ValueError("Input can be either 'start' or 'stop'") # Input validation .

# The following function is used check available models in the local device .
    def available_models(self):
        response=ollama.list() # Getting available models .
        models_available=response.models
        models=[]
        for m in models_available: # Getting required information from model .
            models.append({
                "model_name":m.model,
                "parameters":m.details.parameter_size
            })
        if not models:
            return []

        else:
            return models

# The following function is to check if a specific model is available in local device .
    def is_model_available(self, model_name):
        models = self.available_models()
        return any(m['model_name'] == model_name for m in models)

# The following function is to set model for response .
    def set_model(self):
        models=self.available_models()

        if not models: # Checking if models are available to set .
            print("No models available locally .")
            return
        table=[
            [i,m.get("model_name"),m.get("parameters") ]
            for i,m in enumerate(models)
        ]
        print(tabulate( # Displaying available devices .
            table,
            headers=("Serial","Model Name","Parameters"),
            tablefmt="fancy_grid"
        ))

        while True: # Letting users select the model they want .
            try:
                user_input = input("Type model Serial number (or 'q' to cancel): ").strip()

                if user_input.lower() == 'q':
                    print("Cancelled.")
                    return

                option=int(user_input)

                if 0 <= option < len(models): # Input validation .
                    self.model_name = models[option]['model_name']
                    print(f"Model '{self.model_name}' selected.")
                    return
                else:
                    print(f"Invalid serial. Choose 0-{len(models)-1}")
                    print("Choose valid serial number .")
            except ValueError:
                print("Please enter a valid number.")

# The following function is used to build prompt using user query and retrieved documents .
    def build_prompt(self, query: str, context: str):
        return dedent(f"""
            You are a helpful assistant.
            Answer the question using ONLY the context below.
            If the answer is not present, say "I don't know".

            Context:
            {context}

            Question:
            {query}

            Answer:
        """).strip()

# The following function is used to generate response from language model .
    def generate_response(self,query:str,context:str,stream:bool=True,temperature: float = 0.7,max_tokens: int = 500):
        if not query or not query.strip(): # Query validation .
            raise ValueError("Query cannot be empty")

        if not context or not context.strip(): # Context validation .
            raise ValueError("Context cannot be empty")

        if len(context) > 10000: # Checking if context is too large .
            print("Warning: Large context may be slow")

        try:
            prompt=self.build_prompt(query,context) # Building a prompt using query and context .
            response=ollama.chat( # Getting response from model .
                model=self.model_name,
                messages=[
                    {"role":"system","content":"Answer only using the provided context"},
                    {"role":"user","content":prompt}],
                stream=stream,
                options={
                    'temperature':temperature,
                    "num_predict":max_tokens
                }
            )

            print(f"Query: {query}\nAnswer: ", end="")

            if stream: # Displaying output through streaming .
                full_response = ""
                try:
                    for chunk in response: # Displaying response as model gives output .
                        content = chunk.get("message", {}).get("content", "")
                        if content:
                            print(content, flush=True, end="")
                            full_response+=content
                    print()
                except Exception as e:
                    print(f"\n Error during streaming: {e}") # Exception handling .
                    raise
                return full_response
            else:
                return response["message"]["content"] # If stream is off then giving output all at once .

        except Exception as e:
            raise RuntimeError(f"Error giving response . Error {e}") from e # Exception handling .

In [3]:
llm=LocalLLM() # Initializing model .

Ollama server started


In [4]:
llm.set_model() # Setting models available in local machine .

╒══════════╤══════════════════╤══════════════╕
│   Serial │ Model Name       │ Parameters   │
╞══════════╪══════════════════╪══════════════╡
│        0 │ codellama:7b     │ 7B           │
├──────────┼──────────────────┼──────────────┤
│        1 │ gemma3:4b        │ 4.3B         │
├──────────┼──────────────────┼──────────────┤
│        2 │ deepseek-r1:1.5b │ 1.8B         │
╘══════════╧══════════════════╧══════════════╛
Model 'gemma3:4b' selected.


In [5]:
# Test data .
llm.generate_response(query="Why were light echo images, such as those of V838 Monocerotis, crucial for distinguishing between physical expansion of matter and the propagation of light through surrounding dust?",context="""[1] Expanding Light Echo of V838 Monocerotis
(Source: hubble-science-highlights.pdf, page 37)

[2] In January 2002, an unexplained flash of light from a red
supergiant star left what looked like an expanding bubble of
debris. In fact, the light was simply illuminating clouds that
were already in place around the star. Since light travels at a
finite speed, the flash took years to reach the most distant
clouds and expose them. This phenomenon, called a “light
echo,” is reminiscent of sound waves echoing down a canyon
and “revealing” its walls.
View the V838 Monocerotis movie.
(Source: hubble-science-highlights.pdf, page 37)

[3] Light Echo Around an Exploded Star
(Source: hubble-science-highlights.pdf, page 38)""")

Query: Why were light echo images, such as those of V838 Monocerotis, crucial for distinguishing between physical expansion of matter and the propagation of light through surrounding dust?
Answer: The light echo phenomenon, reminiscent of sound waves echoing down a canyon, revealed clouds that were already in place around the star. This helped distinguish between physical expansion of matter and the propagation of light through surrounding dust.


'The light echo phenomenon, reminiscent of sound waves echoing down a canyon, revealed clouds that were already in place around the star. This helped distinguish between physical expansion of matter and the propagation of light through surrounding dust.'

In [6]:
llm.ollama_server(process="stop") # Stopping server .

Ollama server successfully stopped .
None
